# PS LiDAR - Laboratorio de Desarrollo

Este notebook sirve para probar los **ladrillos** del proyecto de forma interactiva.

**Ladrillos disponibles:**
- Brick 1: Carga de Datos (`PointCloudLoader`)
- Brick 2: Recorte Circular (`clip_circular_plot`)
- Brick 3: Detección de Normalización (`detect_normalization`)
- Brick 4: Filtrado de Suelo (`classify_ground`)

In [ ]:
import os
import sys
import time
from pathlib import Path

# Asegurar que podemos importar desde la carpeta src
module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

# Importar todos los componentes disponibles
from src.core import (
    PointCloudLoader,
    detect_normalization,
    NormalizationStatus,
    classify_ground,
    get_ground_mask,
    get_ground_points,
    clip_circular_plot,
    find_plot_center,
    detect_reflective_mats,
    validate_gps_center,
)

print("✓ Módulos importados correctamente")

---
## 1. Cargar Archivo (Brick 1)
Selecciona un archivo `.las` o `.laz` para empezar.

In [ ]:
# Configura aquí la ruta a tu archivo
FILE_PATH = "../external_references/artemis_treeiso/data/LPine1_demo.laz"

# Cargar
loader = PointCloudLoader(FILE_PATH)
loader.load()

# Mostrar metadatos
meta = loader.get_metadata()
print(f"Archivo: {meta['filename']}")
print(f"Puntos: {meta['point_count']:,}")
print(f"Tamaño: {meta['file_size_mb']} MB")
print(f"Rango Z: {meta['min_coords'][2]:.2f}m a {meta['max_coords'][2]:.2f}m")

In [ ]:
# Obtener coordenadas XYZ completas (para detección de centro)
xyz_full = loader.get_xyz()
print(f"Puntos totales: {len(xyz_full):,}")
print(f"Memoria XYZ: {xyz_full.nbytes / (1024**2):.2f} MB")

---
## 2. Recorte Circular (Brick 2)
Extraer solo los puntos dentro del radio del plot de muestreo.

**Opciones para definir el centro:**
1. Detección automática por mat reflectivo (checkerboard o circular)
2. Coordenadas GPS (solo si la nube está georreferenciada)
3. Centroide geométrico (fallback)

In [ ]:
# Configuración del plot
PLOT_RADIUS = 15.0  # Radio en metros
MAT_SIZE = 0.6      # Tamaño del mat reflectivo en metros

# Opción 1: Detección automática (con intensidad)
try:
    intensity = loader.get_attribute("intensity")
    center_x, center_y, method = find_plot_center(xyz_full, intensity, mat_size=MAT_SIZE)
    print(f"Centro detectado por: {method}")
except:
    # Fallback: centroide geométrico
    center_x, center_y, method = find_plot_center(xyz_full)
    print(f"Centro por: {method}")

print(f"Centro del plot: ({center_x:.2f}, {center_y:.2f})")
print(f"Radio: {PLOT_RADIUS}m")

In [ ]:
# (Opcional) Ver todos los mats detectados
try:
    mats = detect_reflective_mats(xyz_full, intensity, mat_size=MAT_SIZE)
    print(f"Mats detectados: {len(mats)}")
    for i, mat in enumerate(mats):
        print(f"  Mat {i+1}: centro=({mat.center_x:.2f}, {mat.center_y:.2f}), clusters={mat.n_clusters}")
except:
    print("No se pudo detectar mats (¿falta intensidad?)")

In [ ]:
# Ejecutar recorte circular
t0 = time.perf_counter()
clip_result = clip_circular_plot(xyz_full, center_x, center_y, PLOT_RADIUS)
elapsed = time.perf_counter() - t0

print(f"✓ Recorte completado en {elapsed*1000:.1f}ms")
print(f"")
print(f"Puntos originales: {len(xyz_full):,}")
print(f"Puntos en plot:    {clip_result.n_points:,} ({clip_result.n_points/len(xyz_full):.1%})")
print(f"Área del plot:     {clip_result.area_ha:.4f} ha")

In [ ]:
# Extraer datos del plot (XYZ + campos escalares)
plot_indices = clip_result.indices
xyz = xyz_full[plot_indices]  # Coordenadas del plot

# Extraer campos escalares si están disponibles
try:
    plot_intensity = loader.get_attribute("intensity")[plot_indices]
    print(f"✓ Intensidad extraída: {len(plot_intensity):,} valores")
except:
    plot_intensity = None
    print("⚠ Intensidad no disponible")

try:
    plot_returns = loader.get_attribute("number_of_returns")[plot_indices]
    print(f"✓ Número de retornos extraído: {len(plot_returns):,} valores")
except:
    plot_returns = None
    print("⚠ Número de retornos no disponible")

print(f"")
print(f"Memoria del plot (XYZ): {xyz.nbytes / (1024**2):.2f} MB")

---
## 3. Análisis de Normalización (Brick 3)
Verificar si la nube ya tiene las alturas relativas al suelo.

In [ ]:
# Analizar normalización (ahora sobre el plot recortado)
analysis = detect_normalization(xyz)

print(f"Estatus: {analysis.status.value.upper()}")
print(f"Confianza: {analysis.confidence:.1%}")
print(f"¿Normalizada?: {analysis.is_normalized}")
print(f"")
print(f"Estadísticas Z:")
print(f"  Min: {analysis.z_min:.2f}m")
print(f"  Max: {analysis.z_max:.2f}m")
print(f"  Rango: {analysis.z_range:.2f}m")
print(f"  5th percentil: {analysis.percentile_5:.2f}m")
print(f"")
print(f"Razonamiento:")
for reason in analysis.reasoning.split("; "):
    print(f"  • {reason}")

---
## 4. Filtrado de Suelo - CSF (Brick 4)
Separar puntos de suelo de la vegetación usando Cloth Simulation Filter.

In [ ]:
# Ejecutar filtrado de suelo (sobre el plot recortado)
print("Ejecutando Cloth Simulation Filter...")
t0 = time.perf_counter()

result = classify_ground(
    xyz,
    cloth_resolution=1.0,  # Resolución del cloth en metros
    rigidness=1,           # 1=plano, 2=relieve, 3=escarpado
    class_threshold=0.5,   # Distancia umbral para clasificar como suelo
    slope_smooth=True,     # Suavizado para pendientes
)

elapsed = time.perf_counter() - t0
print(f"✓ Completado en {elapsed:.2f}s")
print(f"")
print(f"Resultados:")
print(f"  Suelo: {result.n_ground:,} puntos ({result.ground_ratio:.1%})")
print(f"  Vegetación: {result.n_off_ground:,} puntos ({1 - result.ground_ratio:.1%})")

In [ ]:
# Analizar la distribución Z de los puntos de suelo
ground_z = xyz[result.ground_indices, 2]
veg_z = xyz[result.off_ground_indices, 2]

print("Distribución Z del suelo:")
print(f"  Min: {ground_z.min():.2f}m")
print(f"  Max: {ground_z.max():.2f}m")
print(f"  Media: {ground_z.mean():.2f}m")
print(f"")
print("Distribución Z de vegetación:")
print(f"  Min: {veg_z.min():.2f}m")
print(f"  Max: {veg_z.max():.2f}m")
print(f"  Media: {veg_z.mean():.2f}m")

---
## 5. Próximos Pasos

**Brick 5 (pendiente):** Normalización de Altura
- Usar los puntos de suelo para crear un DTM
- Interpolar altura del terreno para cada punto
- Calcular Z_normalizado = Z - DTM

**Brick 6 (pendiente):** Segmentación de Árboles
- Identificar árboles individuales en la nube normalizada